# Interactive-map layer catalog

Inspect the public map-layer configuration, including WMS sources and rendering parameters used by meteo@uniparthenope applications.

This live demonstration uses public meteo@uniparthenope data. Values are generated
automatically and are intended for research, education, and general information—not for official
warnings, navigation, or safety-critical decisions. Times are UTC. Run and edit the cells in this
browser-based workspace; your changes stay in your browser unless you download the notebook.


In [ ]:
import json
import os
import sys
from urllib.parse import urlencode
from datetime import datetime

if sys.platform == "emscripten":
    import piplite
    await piplite.install("ipywidgets")

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import HTML, clear_output, display

API_BASE = "https://api.meteo.uniparthenope.it" if sys.platform == "emscripten" else os.environ.get(
    "METEO_API_BASE_URL", "https://api.meteo.uniparthenope.it"
).rstrip("/")
BRAND = "#1f4e79"
ACCENT = "#2a91eb"

def api_json(path, params=None):
    url = f"{API_BASE}/{path.lstrip('/')}"
    if params:
        url = f"{url}?{urlencode(params)}"
    if sys.platform == "emscripten":
        from pyodide.http import open_url
        return json.load(open_url(url))
    import requests
    response = requests.get(url, timeout=20)
    response.raise_for_status()
    return response.json()

def forecast_frame(place):
    payload = api_json(f"products/wrf5/timeseries/{place}")
    if payload.get("result") != "ok":
        raise ValueError(f"No WRF forecast is available for {place}")
    frame = pd.DataFrame(payload["timeseries"])
    frame["time"] = pd.to_datetime(frame["dateTime"], format="%Y%m%dZ%H%M", utc=True)
    return frame.set_index("time")

PLACES = {"Gulf of Naples": "ca001", "Gulf of Salerno": "ca002", "Cilento": "ca003"}


In [ ]:
layer = widgets.Dropdown(description="Layer:")
output = widgets.Output()
catalog = {}

def load_catalog():
    global catalog
    catalog = api_json("v2/layers")
    layer.options = sorted(catalog)
    layer.value = layer.options[0]

def render(*_):
    with output:
        clear_output(wait=True)
        try:
            item = catalog[layer.value]
            name = item.get("name", {}).get("en-US", layer.value)
            display(HTML(f"<h2>{name}</h2><p><code>{item.get('type', '')}</code></p>"))
            display(HTML(f"<p style='overflow-wrap:anywhere'><b>Service template:</b> {item.get('url', 'n/a')}</p>"))
            extras = item.get("extras", {})
            display(pd.DataFrame([{"parameter": key, "value": value} for key, value in extras.items()]))
        except Exception as exc:
            display(HTML(f"<b>Layer request failed:</b> {exc}"))

try:
    load_catalog()
    layer.observe(render, names="value")
    display(widgets.VBox([layer, output]))
    render()
except Exception as exc:
    display(HTML(f"<b>Catalog request failed:</b> {exc}"))
